In [1]:
import pandas as pd
import numpy as np

In [2]:
data_path = "../data"
trn_route_name_lookup = pd.read_csv(f"{data_path}/trn_route_name_lookup_ver12.csv")
trn_route_name_lookup = trn_route_name_lookup.drop(columns=["route"])
trn_route_name_lookup.head()

,route_id,route_name,route_shortname,route_longname,usera1,usera2,mode,mode_name,mode_longname,operator,operator_name,direction
0,1,5_1_EA_d0_s1,ACE,NaN,CE,Commuter rail,133,Commuter Rail,ACE,5,ACE,0
1,1,5_1_AM_d0_s1,ACE,NaN,CE,Commuter rail,133,Commuter Rail,ACE,5,ACE,0
2,1,5_1_PM_d1_s2,ACE,NaN,CE,Commuter rail,133,Commuter Rail,ACE,5,ACE,1
3,131,30_131_EA_d1_s359,G,San Francisco - El Cerrito,AC Transit,Express bus,84,Express Bus,AC Transit Transbay,30,AC Transit,1
4,131,30_131_AM_d1_s359,G,San Francisco - El Cerrito,AC Transit,Express bus,84,Express Bus,AC Transit Transbay,30,AC Transit,1


### create summary for routes (except BART and Caltrain)

In [3]:
def create_non_rail_summary(approach, period):
    assignment_file = f"{data_path}/{approach}/assignment/boardings_by_line_{period.lower()}.csv"
    if approach == "tap_approach":
        asgn_result = pd.read_csv(assignment_file)
    else:
        asgn_result = pd.read_csv(assignment_file, sep="\t")

    asgn_result = asgn_result.rename(columns={"line_name": "route_name"}).drop(columns=["mode"])
    asgn_result = pd.merge(asgn_result, trn_route_name_lookup, how="left", on="route_name")

    # remove records with operator 17 (Caltrain) & 26 (BART)
    asgn_result = asgn_result[(asgn_result["operator"] != 17) & (asgn_result["operator"] != 26)]

    # organize to final format
    asgn_result = asgn_result.rename(columns={"operator": "operator_id", "operator_name": "operator", "total_boardings": "boardings"})
    asgn_result["source"] = approach
    asgn_result["period"] = period
    asgn_result = asgn_result[["source", "period", "operator_id", "operator", "route_id", "route_shortname", "route_longname", "boardings"]]

    return asgn_result

In [4]:
non_rail_summary = pd.DataFrame()

for approach in ["tap_approach", "taz_approach"]:
    for period in ["EA", "AM", "MD", "PM", "EV"]:
        non_rail_period_approach_summary = create_non_rail_summary(approach, period)
        non_rail_summary = pd.concat([non_rail_summary, non_rail_period_approach_summary])

In [5]:
# reformat
non_rail_summary = non_rail_summary.reset_index(drop=True)
non_rail_summary.loc[non_rail_summary["route_shortname"].isnull(), "route"] = non_rail_summary["route_longname"]
non_rail_summary.loc[non_rail_summary["route_longname"].isnull(), "route"] = non_rail_summary["route_shortname"]
non_rail_summary.loc[(~non_rail_summary["route_shortname"].isnull()) & (~non_rail_summary["route_longname"].isnull()), "route"] = non_rail_summary["route_shortname"].astype(str) + "_" + non_rail_summary["route_longname"].astype(str)
non_rail_summary["stop_name"] = np.nan
non_rail_summary = non_rail_summary[["source", "period", "operator_id", "operator", "route_id", "route", "stop_name", "boardings"]]
non_rail_summary.head()

,source,period,operator_id,operator,route_id,route,stop_name,boardings
0,tap_approach,EA,10,Emery Go-Round,198,Hollis,NaN,0.0
1,tap_approach,EA,10,Emery Go-Round,201,Shellmound/Powell,NaN,25.2
2,tap_approach,EA,12,Vallejo Transit,481,1,NaN,0.0
3,tap_approach,EA,12,Vallejo Transit,481,1,NaN,0.0
4,tap_approach,EA,12,Vallejo Transit,482,2,NaN,0.0


### create summary for BART and Caltrain

In [6]:
bart_nodes = pd.read_csv(f"{data_path}/bart_nodes_with_name.csv")
bart_nodes = bart_nodes.rename(columns={"N": "model_node_id", "stop_nm": "stop_name"})
bart_nodes = bart_nodes[["model_node_id", "stop_name"]]
bart_nodes.head()

,model_node_id,stop_name
0,1027608,Embarcadero
1,1027609,Montgomery St.
2,1027610,Powell St.
3,1027611,Civic Center/UN Plaza
4,1027612,16th St. Mission


In [7]:
def create_rail_summary(approach, operator, period):
    if operator == "Caltrain":
        operator_id = 17
    elif operator == "BART":
        operator_id = 26
    
    route_id_name = {
        "route_id": [154, 155, 156, 157, 158, 159, 193, 194, 195],
        "route": [
            "Pittsburg/Bay Point - SFIA/Millbrae",
            "Fremont - Richmond",
            "Fremont - Daly City",
            "Richmond - Daly City/Millbrae",
            "Dublin/Pleasanton - Daly City",
            "Coliseum - Oakland Int'l Airport",
            "Bullet",
            "Limited",
            "Local",
    ]}
    route_id_name_df = pd.DataFrame.from_dict(route_id_name)

    if approach == "tap_approach":
        node_asgn_result = pd.read_csv(f"{data_path}/{approach}/assignment/boardings_by_node_{period.lower()}.csv", sep="\t")
    elif approach == "taz_approach":
        node_asgn_result = pd.read_csv(f"{data_path}/{approach}/assignment/boardings_by_segment_{period.lower()}.csv", sep="\t")
    node_asgn_result = node_asgn_result[node_asgn_result["is_i_node_stop"] == 1]
    node_asgn_result = node_asgn_result.rename(columns={"line_name": "route_name"})
    node_asgn_result = pd.merge(node_asgn_result, trn_route_name_lookup, how="left", on="route_name")
    node_asgn_result = node_asgn_result.rename(columns={"operator": "operator_id", "operator_name": "operator"})
    node_asgn_result = node_asgn_result[node_asgn_result["operator_id"] == operator_id]

    node_asgn_result = pd.merge(node_asgn_result, route_id_name_df, how="left", on="route_id")

    node_asgn_result["source"] = approach
    node_asgn_result["period"] = period

    node_asgn_result = node_asgn_result[["source", "period", "operator_id", "operator", "route_id", "route", "stop_name", "boardings"]]
    node_asgn_result = node_asgn_result.groupby(["source", "period", "operator_id", "operator", "route_id", "route", "stop_name"])["boardings"].agg("sum").reset_index(name="boardings")
    
    return node_asgn_result

In [8]:
bart_summary = pd.DataFrame()

for approach in ["tap_approach", "taz_approach"]:
    for period in ["EA", "AM", "MD", "PM", "EV"]:
        bart_period_approach_summary = create_rail_summary(approach, "BART", period)
        bart_summary = pd.concat([bart_summary, bart_period_approach_summary])

bart_summary.head()

,source,period,operator_id,operator,route_id,route,stop_name,boardings
0,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,12th St. Oakland City Center,56.188333
1,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,16th St. Mission,54.182210
2,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,19th St. Oakland,61.335000
3,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,24th St. Mission,45.117010
4,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,Balboa Park,74.661820


In [9]:
caltrain_summary = pd.DataFrame()

for approach in ["tap_approach", "taz_approach"]:
    for period in ["EA", "AM", "MD", "PM", "EV"]:
        caltrain_period_approach_summary = create_rail_summary(approach, "Caltrain", period)
        caltrain_summary = pd.concat([caltrain_summary, caltrain_period_approach_summary])

caltrain_summary.head()

,source,period,operator_id,operator,route_id,route,stop_name,boardings
0,tap_approach,EA,17,Caltrain,193,Bullet,Hillsdale Caltrain,0.00
1,tap_approach,EA,17,Caltrain,193,Bullet,Millbrae Caltrain,0.00
2,tap_approach,EA,17,Caltrain,193,Bullet,Mt View Caltrain,2.99
3,tap_approach,EA,17,Caltrain,193,Bullet,Palo Alto Caltrain,0.00
4,tap_approach,EA,17,Caltrain,193,Bullet,San Francisco Caltrain,0.00


In [10]:
result_long = pd.concat([bart_summary, caltrain_summary, non_rail_summary])
result_long.head()

,source,period,operator_id,operator,route_id,route,stop_name,boardings
0,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,12th St. Oakland City Center,56.188333
1,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,16th St. Mission,54.182210
2,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,19th St. Oakland,61.335000
3,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,24th St. Mission,45.117010
4,tap_approach,EA,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,Balboa Park,74.661820


In [ ]:
result_long.to_csv("../../outputs/assignment/boarding_comparison_v6.csv", index=False)

## reshape result from long to wide format

In [11]:
result_wide_bart = pd.pivot_table(bart_summary, values="boardings", 
                                                index=["period", "operator_id", "operator", "route_id", "route", "stop_name"],
                                                columns=["source"],
                                                aggfunc="sum").reset_index().rename(columns={"tap_approach": "tap_boardings",
                                                                                             "taz_approach": "taz_boardings"})
result_wide_bart.head()

source,period,operator_id,operator,route_id,route,stop_name,tap_boardings,taz_boardings
0,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,12th St. Oakland City Center,1200.9943,810.6908
1,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,16th St. Mission,582.8108,392.6731
2,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,19th St. Oakland,1065.1420,238.4674
3,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,24th St. Mission,1190.6400,701.2632
4,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,Balboa Park,1301.0640,609.8207


In [12]:
result_wide_caltrain = pd.pivot_table(caltrain_summary, values="boardings", 
                                                        index=["period", "operator_id", "operator", "route_id", "route", "stop_name"],
                                                        columns=["source"],
                                                        aggfunc="sum").reset_index().rename(columns={"tap_approach": "tap_boardings",
                                                                                                     "taz_approach": "taz_boardings"})
result_wide_caltrain.head()

source,period,operator_id,operator,route_id,route,stop_name,tap_boardings,taz_boardings
0,AM,17,Caltrain,193,Bullet,22nd St Caltrain,282.52500,756.62500
1,AM,17,Caltrain,193,Bullet,Hillsdale Caltrain,18.50638,721.96240
2,AM,17,Caltrain,193,Bullet,Menlo Park Caltrain,18.71667,53.86667
3,AM,17,Caltrain,193,Bullet,Millbrae Caltrain,258.64738,714.91720
4,AM,17,Caltrain,193,Bullet,Mt View Caltrain,400.91035,278.11073


In [13]:
result_wide_nonrail = pd.pivot_table(non_rail_summary, values="boardings", 
                                          index=["period", "operator_id", "operator", "route_id", "route"],
                                          columns=["source"],
                                          aggfunc="sum").reset_index().rename(columns={"tap_approach": "tap_boardings",
                                                                                       "taz_approach": "taz_boardings"})
result_wide_nonrail.head()

source,period,operator_id,operator,route_id,route,tap_boardings,taz_boardings
0,AM,1,Santa Rosa CityBus,464,9_9 Sebastopol Rd,157.33550,4.650000
1,AM,1,Santa Rosa CityBus,465,10_10 Coddingtown,30.25169,4.227143
2,AM,1,Santa Rosa CityBus,466,11_11 Fulton Rd,63.96000,1.253333
3,AM,1,Santa Rosa CityBus,467,12_12 Roseland,85.72968,0.485161
4,AM,1,Santa Rosa CityBus,468,14_14 County Center,170.66180,18.825340


In [14]:
result_wide = pd.concat([result_wide_bart, result_wide_caltrain, result_wide_nonrail])
result_wide.head()

source,period,operator_id,operator,route_id,route,stop_name,tap_boardings,taz_boardings
0,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,12th St. Oakland City Center,1200.9943,810.6908
1,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,16th St. Mission,582.8108,392.6731
2,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,19th St. Oakland,1065.1420,238.4674
3,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,24th St. Mission,1190.6400,701.2632
4,AM,26,BART,154,Pittsburg/Bay Point - SFIA/Millbrae,Balboa Park,1301.0640,609.8207


In [ ]:
result_wide.to_csv("../../outputs/assignment/boarding_comparison_v6_wide.csv", index=False)